# Samudrika – Underwater Image Enhancement (Colab)

This notebook runs the full Samudrika pipeline: install deps, load UIEB, optionally train, run inference, and compute PSNR, SSIM, UIQM.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Install dependencies

In [1]:
!pip install -q torch torchvision numpy Pillow scikit-image matplotlib

## 2. Clone repository (or upload project)

In [ ]:
# Option A: Clone your repo (replace with your repo URL)
# !git clone https://github.com/your-org/Samudrika.git
# %cd Samudrika

# Option B: If you uploaded the project via Colab file browser, just go to the project folder
%cd /content
import os
if not os.path.exists('Samudrika'):
    os.makedirs('Samudrika', exist_ok=True)
    print('Upload the Samudrika project files to /content/Samudrika or clone the repo above.')

## 3. Mount Google Drive (optional)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Example: use a dataset or checkpoints from Drive
# DATA_ROOT = '/content/drive/MyDrive/UIEB'
# CHECKPOINT = '/content/drive/MyDrive/checkpoints/samudrika_epoch_100.pt'

## 4. Load UIEB dataset

In [ ]:
import sys
sys.path.insert(0, '/content/Samudrika')
sys.path.insert(0, '/content/Samudrika/samudrika')

from samudrika.dataset import get_uieb_dataloaders

DATA_ROOT = '/content/Samudrika/data/UIEB'  # or your path, e.g. /content/drive/MyDrive/UIEB
train_loader, val_loader = get_uieb_dataloaders(DATA_ROOT, batch_size=8, num_workers=0, size=256)
print('Train batches:', len(train_loader))
if val_loader:
    print('Val batches:', len(val_loader))

# Show one batch
raw, ref = next(iter(train_loader))
print('Raw shape:', raw.shape, 'Ref shape:', ref.shape)

## 5. Train model (optional)

In [ ]:
# Short run for demo (increase epochs for real training)
!cd /content/Samudrika && python train_funie.py --data_root {DATA_ROOT} --epochs 5 --batch_size 4 --save_every 2 --save_dir ./checkpoints

## 6. Run inference on test images

In [ ]:
from pathlib import Path
from samudrika import run_pipeline

CHECKPOINT = '/content/Samudrika/checkpoints/samudrika_epoch_5.pt'  # or your latest checkpoint
if not Path(CHECKPOINT).exists():
    CHECKPOINT = '/content/Samudrika/checkpoints/samudrika_epoch_100.pt'  # fallback

# Use first image from dataset as test, or set TEST_IMAGE to any path
TEST_IMAGE = None
if train_loader and len(train_loader.dataset) > 0:
    from samudrika.dataset import UIEBDataset
    ds = train_loader.dataset
    TEST_IMAGE = str(ds.pairs[0][0])  # first raw image
    print('Test image:', TEST_IMAGE)
else:
    print('Upload a test image or set TEST_IMAGE to a path.')

if TEST_IMAGE and Path(TEST_IMAGE).exists():
    result = run_pipeline(TEST_IMAGE, CHECKPOINT, output_path='/content/out_enhanced.jpg', compute_metrics=True, display=True)
    print('Metrics:', result.get('metrics', {}))

## 7. Display enhancement results

In [ ]:
import matplotlib.pyplot as plt
from samudrika.utils import load_image

if TEST_IMAGE and Path(TEST_IMAGE).exists() and 'result' in dir():
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(load_image(TEST_IMAGE))
    axes[0].set_title('Input (underwater)')
    axes[0].axis('off')
    axes[1].imshow(result['enhanced'])
    axes[1].set_title('Enhanced')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

## 8. Compute PSNR, SSIM, UIQM

In [ ]:
from samudrika.metrics import psnr, ssim, uiqm
import numpy as np

if TEST_IMAGE and Path(TEST_IMAGE).exists() and 'result' in dir():
    enhanced = result['enhanced']
    print('UIQM (enhanced, no reference):', round(uiqm(enhanced), 4))
    if result.get('psnr') is not None:
        print('PSNR:', round(result['psnr'], 2), 'dB')
        print('SSIM:', round(result['ssim'], 4))
    else:
        print('For PSNR/SSIM, run run_pipeline(..., reference_path="path/to/reference.jpg")')